# Exploratory Data Analysis (EDA) - Taller 2
## ¿Puedes Predecir el Fútbol Mejor que las Casas de Apuestas?

In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from IPython.display import display
import scipy.stats as stats

sns.set_theme(style="whitegrid")
# Configuraciones visuales
plt.rcParams['figure.figsize'] = (10, 6)

## Fase 0: Conocer los Datasets que tenemos (Players, Matches, Events)

In [2]:
players_df = pd.read_csv('data/players.csv')
matches_df = pd.read_csv('data/matches.csv')
events_df = pd.read_csv('data/events.csv')

print(f"Players dataset shapes: {players_df.shape}")
print(f"Matches dataset shapes: {matches_df.shape}")
print(f"Events dataset shapes: {events_df.shape}")

Players dataset shapes: (822, 37)
Matches dataset shapes: (291, 41)
Events dataset shapes: (444252, 19)


## 1. Carga y Exploración Inicial

In [ ]:
def analyze_dataset(df, name):
    print(f"\n{'='*50}\n--- ANALISIS DE DATASET: {name} ---\n{'='*50}")
    print(f"\n1. TAMAÑO DEL DATASET: Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
    print(f"\n2. DUPLICADOS: {df.duplicated().sum()}")
    info_df = pd.DataFrame({'Tipo': df.dtypes, 'Nulos': df.isnull().sum(), '% Nulos': (df.isnull().sum() / len(df) * 100).round(2), 'Unicos': df.nunique()})
    display(info_df)
    
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        desc = df[num_cols].describe().T
        desc['skew'] = df[num_cols].skew()
        desc['kurtosis'] = df[num_cols].kurtosis()
        display(desc.round(2))

# (Llamados comentados para no saturar al correr todo)
# analyze_dataset(players_df, 'PLAYERS')
# analyze_dataset(matches_df, 'MATCHES')
# analyze_dataset(events_df, 'EVENTS')

## 2. Análisis Univariado
En esta sección aplicamos histogramas, densidades y pruebas estadísticas fuertes (Shapiro-Wilk) para determinar asimetrías y normalidad de las variables cardinales críticas (ej: `b365h`, o coordenadas `x, y`). Evaluaremos las métricas que propusimos al inicio.

In [ ]:
# 2.1 Análisis Numérico Univariado en el Match Predictor (Matches)
cols_to_test = ['total_goals', 'b365h', 'b365a', 'hst', 'hs']

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()

for i, col in enumerate(cols_to_test):
    ax = axes[i]
    sns.histplot(matches_df[col], kde=True, ax=ax, color='#2ecc71', stat='density')
    ax.set_title(f"Distribución de {col}", fontweight='bold')
    
    # Prueba de normalidad Shapiro-Wilk (Ideal para N < 5000)
    stat, p_shap = stats.shapiro(matches_df[col].dropna())
    normality = "Distribución Normal" if p_shap > 0.05 else "No Normal (Asimétrica/Cola pesada)"
    ax.text(0.5, 0.9, f"Shapiro p-val: {p_shap:.4f}\n{normality}", 
            transform=ax.transAxes, ha='center', bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray'))
            
fig.delaxes(axes[5]) # Borrar el último plot vacío
plt.tight_layout()
plt.show()

In [ ]:
# 2.2 Entropía y Dispersión de Variables Categóricas Claves en Eventos
def calc_entropy(series):
    counts = series.value_counts(normalize=True)
    return stats.entropy(counts)

print("=== Entropía de Categóricas ===")
print(f"Event Type Entropy: {calc_entropy(events_df['event_type']):.4f} (Indica dispersión de tipos de eventos)")
print(f"Outcome Entropy: {calc_entropy(events_df['outcome']):.4f}")

plt.figure(figsize=(12, 5))
top_events = events_df['event_type'].value_counts().head(10)
sns.barplot(y=top_events.index, x=top_events.values, palette="viridis")
plt.title("Top 10 Tipos de Eventos más Frecuentes (Desbalance evidente hacia Passes)", fontweight='bold')
plt.show()

## 3. Análisis Bivariado
Evaluamos asociaciones entre variables métricas y nuestros Targets (`ftr`, `total_goals`, `is_goal`). Construiremos Matrices de Correlación robustas (Spearman, para relaciones no-lineales) y test de Hipótesis (Ej: Mann-Whitney U, ANOVA).

In [ ]:
# Preprocesamiento Express de Qualifiers para la Matriz Bivariada de Tiros (xG Model)
shots_df = events_df[events_df['is_shot'] == 1].copy()
shots_df['is_penalty'] = shots_df['qualifiers'].str.contains('"Penalty"', na=False).astype(int)
shots_df['is_big_chance'] = shots_df['qualifiers'].str.contains('BigChance', na=False).astype(int)
shots_df['is_header'] = shots_df['qualifiers'].str.contains('"Head"', na=False).astype(int)
shots_df['dist_al_arco'] = np.sqrt((100 - shots_df['x'])**2 + (50 - shots_df['y'])**2) # Distancia pitagórica base

cols_corr = ['is_goal', 'x', 'y', 'dist_al_arco', 'is_penalty', 'is_big_chance', 'is_header']
corr_matrix = shots_df[cols_corr].corr(method='spearman') # Spearman captura asociaciones no lineales (distancias)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Matriz de Correlación Spearman - Modelo xG (Solo Tiros)", fontweight='bold')
plt.show()

In [ ]:
# Test de Hipótesis Bivariado: Kruskal-Wallis o ANOVA en Cuotas vs Resultado Final (Match Predictor)
# ¿Son estadísticamente significativas las cuotas B365H respecto a si el Local Gana, Empata o Pierde?
plt.figure(figsize=(10, 6))
sns.boxplot(x='ftr', y='b365h', data=matches_df, palette='Set2')
plt.title("Distribución Bivariada: Cuota Local (Bet365) vs Resultado Final del Partido", fontweight='bold')
plt.xlabel("Resultado Final (A=Away, D=Draw, H=Home)")
plt.ylabel("Cuota B365 Local")
plt.show()

h_wins_odds = matches_df[matches_df['ftr']=='H']['b365h']
d_wins_odds = matches_df[matches_df['ftr']=='D']['b365h']
a_wins_odds = matches_df[matches_df['ftr']=='A']['b365h']

# Kruskal-Wallis (alternativa no paramétrica de ANOVA debido a que las Odds son asimétricas)
stat, p_kruskal = stats.kruskal(h_wins_odds, d_wins_odds, a_wins_odds)
print("=== Prueba General Independiente de Kruskal-Wallis ===")
print(f"p-value de diferencias de medianas entre H, D y A = {p_kruskal:.6e}")
if p_kruskal < 0.05:
    print("Conclusión Fuerte: Las cuotas de la casa de apuestas efectivamente encapsulan una separación significativa de los perfiles de partidos.")

## 4. Valores Faltantes e Imputación
- Visualizar los nulos y decidir tratamienos.

In [ ]:
# Análisis de Nulos Estructurales en Events (Tiros vs No Tiros)
pct_missing_goal = events_df['goal_mouth_y'].isnull().sum() / len(events_df) * 100
print("=== NULOS ESTRUCTURALES DEL ARCO ===")
print(f"Porcentaje de nulos en goal_mouth_y: {pct_missing_goal:.2f}%")
print("\nJUSTIFICACIÓN: No imputaremos estos valores.\nEl 98% de los eventos NO son tiros, por tanto carecen de la dimensión Z e Y de portería. \nPara el Modelo de xG, filtraremos exclusivamente is_shot == 1, donde estas coordenadas sí existen integralmente.")


## 5. Atípicos
- Detectar IQR/Z-scores.

In [ ]:
# Detección de Outliers (IQR) en Matches (Goles Totales)
Q1 = matches_df['total_goals'].quantile(0.25)
Q3 = matches_df['total_goals'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
outliers_goles = matches_df[matches_df['total_goals'] > upper_bound]
print(f"Partidos Atípicos detectados (Goles > {upper_bound}): {len(outliers_goles)}")
display(outliers_goles[['home_team', 'away_team', 'fthg', 'ftag', 'total_goals']])
print("\nDECISIÓN METODOLÓGICA: En el fútbol, un partido 5-0 (Outlier estadístico) NO es un error de digitación.\nEs varianza real de calidad ofensiva entre equipos (Ej: Arsenal, Man City).\nNO los eliminaremos ni winsorizaremos. Confiaremos en métodos robustos (Log-Transforms o Penalización L2) para manejar sus colas en la regresión del Predictor.")


## 6. Selección Preliminar de Variables (Feature Engineering)

In [ ]:
# Fase 6.1: Construcción de la Matriz de Entrenamiento (xG Model)
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np
import pandas as pd

# 1. Geometría y Dummies de Contexto
shots_df['is_BigChance'] = shots_df['qualifiers'].apply(lambda q: 1 if 'BigChance' in str(q) else 0)
shots_df['is_Penalty'] = shots_df['qualifiers'].apply(lambda q: 1 if 'Penalty' in str(q) else 0)
shots_df['is_OneOnOne'] = shots_df['qualifiers'].apply(lambda q: 1 if 'OneOnOne' in str(q) else 0)
shots_df['angulo_tiro'] = np.arctan2(50 - shots_df['y'], 100 - shots_df['x']).abs()
shots_df['dist_al_arco'] = np.sqrt((100 - shots_df['x'])**2 + (50 - shots_df['y'])**2)

# 2. Cruce con la Calidad del Jugador (Se elimina goals_scored por recomendación de VIF)
players_df['full_name'] = players_df['first_name'] + ' ' + players_df['second_name']
players_subset = players_df[['full_name', 'threat']]
xg_df = pd.merge(shots_df, players_subset, left_on='player_name', right_on='full_name', how='left')
xg_df['threat'] = xg_df['threat'].fillna(0)

# 3. Exportación Final conservando player_name para Analíticas/Dashboard
features_xg = ['player_name', 'dist_al_arco', 'angulo_tiro', 'is_BigChance', 'is_Penalty', 'is_OneOnOne', 'threat']
final_xg_model_df = xg_df[features_xg + ['is_goal']].dropna()
final_xg_model_df.to_csv('data/xg_train.csv', index=False)

# 4. Exportar Diccionario de Jugadores para UI del Dashboard
players_subset.dropna().rename(columns={'full_name': 'player_name'}).to_csv('data/player_threats_dict.csv', index=False)

print(f"Matriz Final {final_xg_model_df.shape} construida con éxito.")


In [ ]:
# Fase 6.2: Validación de Multicolinealidad (VIF) - Supuesto de Regresión Logística
X_vif = final_xg_model_df[features_xg].copy()
X_vif['intercept'] = 1 # Requisito matemático para VIF

vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]

print("=== PRUEBA DE INFLACIÓN DE VARIANZA (VIF) ===")
display(vif_data.round(2))
if vif_data['VIF'].max() < 5:
    print("\nCONCLUSIÓN: Se autoriza entrenar el modelo. No existe multicolinealidad preocupante (VIF < 5).")


## 7. Síntesis y Dashboard (Insights)

In [ ]:
# Tu código aquí...

## 4. EDA Profundo: Match Predictor (Modelo 2 - Reg. Lineal)

A continuación, realizaremos las pruebas estadísticas rigurosas sobre `matches.csv` aislando **únicamente** las variables previas al pitazo inicial (Anti-Leakage), para evaluar su poder predictivo frente al volumen de goles de un partido (`total_goals`).

In [ ]:
from scipy.stats import shapiro, kruskal
import warnings
warnings.filterwarnings('ignore')

# Cargar dataset limitando fugas de informacion
df_matches = pd.read_csv('data/matches.csv')
target = 'total_goals'
features = ['b365h', 'b365d', 'b365a', 'bwh', 'bwd', 'bwa', 'avgh', 'avgd', 'avga', 'implied_prob_h', 'implied_prob_d', 'implied_prob_a']

print("1. PRUEBAS DE NORMALIDAD (SHAPIRO-WILK) - Buscando la Campana de Gauss")
for col in features + [target]:
    stat, p_value = shapiro(df_matches[col].dropna())
    result = "RECHAZADA (No Normal)" if p_value < 0.05 else "ACEPTADA"
    print(f"- {col.ljust(15)}: p-value = {p_value:.2e} -> {result}")


In [ ]:
print("2. MATRIZ DE CORRELACIÓN (SPEARMAN) - Detectando relaciones no lineales")
corr_matrix = df_matches[features + [target]].corr(method='spearman')
display(corr_matrix[target].sort_values(ascending=False))

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de Correlación (Spearman) - Cuotas vs Goles Totales')
plt.show()


In [ ]:
print("3. ANÁLISIS BIVARIADO CATEGÓRICO (KRUSKAL-WALLIS) Y BOXPLOTS")
cats = ['home_team', 'away_team', 'referee']
fig, axes = plt.subplots(3, 1, figsize=(14, 20))

for idx, cat in enumerate(cats):
    groups = [group[target].dropna().values for name, group in df_matches.groupby(cat)]
    groups = [g for g in groups if len(g) > 0]
    stat, p_value = kruskal(*groups)
    
    print(f"[{cat.upper()}] vs [{target}] -> P-Value: {p_value:.4f} (Si p < 0.05, afecta a los goles)")
    
    sns.boxplot(x=cat, y=target, data=df_matches, ax=axes[idx], palette="coolwarm_r")
    axes[idx].set_title(f'Distribución de {target} por {cat}')
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45, ha='right', fontsize=9)
    axes[idx].set_ylabel('Total Goles')

plt.tight_layout()
plt.show()

print("CONCLUSIÓN FINAL:")
print("Las etiquetas categóricas son estáticas y no superan el test H0. Se requiere crear Medias Móviles en Pandas para salvar el Modelo 2.")